
# 06 - End-to-End Assistant

## Goal

Connect the complete Acme Motors Assistant flow.

The assistant will:

1. Receive a natural language business question.
2. Use a Databricks Foundation Model through ai_query to select a safe function.
3. Validate the router output.
4. Execute the function through FunctionRegistry.
5. Query the auto_sales_transactions Delta Table.
6. Return a final answer based on real data.
7. Save the full interaction into an audit Delta Table.

## Full flow

User question  
↓  
Databricks Foundation Model Router  
↓  
Structured JSON function call  
↓  
FunctionRegistry validation  
↓  
Controlled SQL query  
↓  
auto_sales_transactions Delta Table  
↓  
Final answer  
↓  
assistant_query_audit Delta Table  

## Example

User:

¿Cuántas F150 he vendido este mes?

↓

Foundation model output:

{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

↓

FunctionRegistry executes:

get_product_sales_summary(product_model="F150", date_range="this_month")

↓

Answer:

For this_month, F150 sold 5 units with $266,500.00 total revenue.

## Why this prevents hallucinations

The model does not calculate sales numbers.  
The model does not generate SQL.  
The model does not access arbitrary tables.  

The model only routes the question.  
Python executes controlled functions.  
Delta Tables provide the source of truth.

In [0]:
%run ./04_python_function_registry


# 04 - Python Function Registry

## Goal

Create the Python object that executes the safe business functions described in the function catalog.

The foundation model will eventually return JSON like:

{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

But the model will NOT execute SQL.

The flow is:

Function call JSON  
↓  
Validate function name  
↓  
Validate arguments  
↓  
Execute approved Python method  
↓  
Run controlled query against Delta Table  
↓  
Return structured result  

## Example

User question:

¿Cuántas F150 vendí este mes?

↓

Foundation model output later:

{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

↓

FunctionRegistry executes:

registry.execute(
  function_name="get_product_sales_summary",
  arguments={
    "product_model": "F150",
    "date_range": "this_month"
  }
)

↓

The registry validates:

- function exists
- product_model exists in auto_sales_transactions
- date_range is allowed

↓

Then it runs a controlled query.

↓

Answer:

This month, F150 sold 5 units with $266,500 total revenue.

## Current step

We are here:

Function catalog  
↓  
Python FunctionRegistry  
↓  
Safe query execution  

## Output of this notebook

A working FunctionRegistry class that can execute approved business functions safely.

Source table: auto_sales_transactions
Function catalog table: assistant_function_catalog
Today: 2026-05-23


sale_id,sale_date,dealership_id,dealership_name,brand,product_model,vehicle_type,sales_rep,customer_state,quantity,sale_price,discount_amount,financing_type,created_at
S001,2026-05-01,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,52000.0,1500.0,Finance,2026-05-23T21:11:25.384Z
S002,2026-05-03,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,53500.0,1000.0,Lease,2026-05-23T21:11:25.384Z
S003,2026-05-05,D002,Acme Motors South,Chevrolet,Silverado,Truck,Luis Garcia,CA,1,49000.0,1200.0,Finance,2026-05-23T21:11:25.384Z
S004,2026-05-07,D003,Acme Motors East,Ford,Mustang,Coupe,Mike Johnson,AZ,1,44000.0,800.0,Cash,2026-05-23T21:11:25.384Z
S005,2026-05-08,D002,Acme Motors South,Toyota,Camry,Sedan,Luis Garcia,CA,1,31000.0,500.0,Finance,2026-05-23T21:11:25.384Z
S006,2026-05-10,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,54000.0,900.0,Finance,2026-05-23T21:11:25.384Z
S007,2026-05-11,D003,Acme Motors East,Honda,Civic,Sedan,Sarah Kim,NV,1,28000.0,300.0,Lease,2026-05-23T21:11:25.384Z
S008,2026-05-13,D002,Acme Motors South,Chevrolet,Silverado,Truck,Luis Garcia,CA,1,50500.0,1000.0,Finance,2026-05-23T21:11:25.384Z
S009,2026-05-15,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,52500.0,1100.0,Cash,2026-05-23T21:11:25.384Z
S010,2026-05-17,D003,Acme Motors East,Ford,Explorer,SUV,Mike Johnson,AZ,1,47000.0,700.0,Finance,2026-05-23T21:11:25.384Z


function_name,description,source_table
get_product_sales_summary,"Returns units sold, total revenue, and average sale price for a specific vehicle model in a date range.",auto_sales_transactions
get_top_selling_models,Returns the top selling vehicle models by units sold and revenue for a date range.,auto_sales_transactions
get_sales_by_rep,Returns units sold and revenue grouped by sales representative for a date range.,auto_sales_transactions
get_revenue_by_state,Returns units sold and revenue grouped by customer state for a date range.,auto_sales_transactions
get_average_sale_price,"Returns average, minimum, and maximum sale price for a specific vehicle model in a date range.",auto_sales_transactions
get_dealership_sales_summary,"Returns units sold, revenue, and average sale price grouped by dealership for a date range.",auto_sales_transactions



## What is a Function Registry?

A Function Registry is an object that knows which functions are allowed and how to execute them.

It works like a controlled dispatcher.

Instead of allowing the model to run arbitrary SQL, we only allow this:

{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

Then the registry decides:

- Is this function allowed?
- Are the arguments valid?
- Which method should run?
- Which Delta table should be queried?

This is how we reduce hallucination risk.

The LLM routes.  
Python executes.  
Delta is the source of truth.

In [0]:
%run ./05_foundation_model_router

this_month => 2026-05-01 to 2026-05-23
last_month => 2026-04-01 to 2026-04-30
year_to_date => 2026-01-01 to 2026-05-23



# 05 - Foundation Model Router

## Goal

Use a Databricks Foundation Model as a function router.

The model will receive:

- the user question
- the approved function catalog
- examples of expected JSON outputs

The model must NOT answer the business question directly.

The model must only return JSON like:

{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

## Current flow

User question  
↓  
Prompt builder  
↓  
Databricks Foundation Model  
↓  
Structured JSON  
↓  
Validated router output  

## Next step

In the next notebook, the router output will be passed into the FunctionRegistry.

The FunctionRegistry will execute the safe query against the Delta Table.

## Why this matters

This prevents hallucinations because:

- the model does not calculate sales numbers
- the model does not generate SQL
- the model only selects from approved functions
- Python validates the selected function and arguments
- Delta Tables remain the source of truth

Available functions:
['get_product_sales_summary', 'get_top_selling_models', 'get_sales_by_rep', 'get_revenue_by_state', 'get_average_sale_price', 'get_dealership_sales_summary']

Available models:
['Accord', 'Camry', 'Civic', 'Explorer', 'F150', 'Mustang', 'RAV4', 'Silverado']

Allowed date ranges:
['this_month', 'last_month', 'year_to_date']


Function catalog table: assistant_function_catalog
Model endpoint: databricks-meta-llama-3-3-70b-instruct


function_name,description,arguments_schema_json,example_questions_json,expected_json_example
get_product_sales_summary,"Returns units sold, total revenue, and average sale price for a specific vehicle model in a date range.","{""product_model"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""Accord"", ""Camry"", ""Civic"", ""Explorer"", ""F150"", ""Mustang"", ""RAV4"", ""Silverado""]}, ""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}}","[""How many F150 trucks did I sell this month?"", ""Cu\u00e1ntas F150 he vendido este mes?"", ""What revenue did Silverado generate last month?"", ""Dame el resumen de ventas de Mustang este mes""]","{""function_name"": ""get_product_sales_summary"", ""arguments"": {""product_model"": ""F150"", ""date_range"": ""this_month""}}"
get_top_selling_models,Returns the top selling vehicle models by units sold and revenue for a date range.,"{""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}, ""limit"": {""type"": ""integer"", ""required"": false, ""default"": 5, ""min"": 1, ""max"": 10}}","[""What are my top selling models this month?"", ""Cu\u00e1les son mis modelos m\u00e1s vendidos este mes?"", ""Show me the top 5 models year to date""]","{""function_name"": ""get_top_selling_models"", ""arguments"": {""date_range"": ""this_month"", ""limit"": 5}}"
get_sales_by_rep,Returns units sold and revenue grouped by sales representative for a date range.,"{""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}}","[""Which sales rep sold the most this month?"", ""Qu\u00e9 vendedor vendi\u00f3 m\u00e1s este mes?"", ""Show sales performance by representative year to date""]","{""function_name"": ""get_sales_by_rep"", ""arguments"": {""date_range"": ""this_month""}}"
get_revenue_by_state,Returns units sold and revenue grouped by customer state for a date range.,"{""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}}","[""How much revenue did we generate by state this month?"", ""Cu\u00e1nto vendimos por estado este mes?"", ""Show revenue by state year to date""]","{""function_name"": ""get_revenue_by_state"", ""arguments"": {""date_range"": ""this_month""}}"
get_average_sale_price,"Returns average, minimum, and maximum sale price for a specific vehicle model in a date range.","{""product_model"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""Accord"", ""Camry"", ""Civic"", ""Explorer"", ""F150"", ""Mustang"", ""RAV4"", ""Silverado""]}, ""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}}","[""What was the average sale price for Mustang this month?"", ""Cu\u00e1l fue el ticket promedio de Mustang este mes?"", ""Average sale price for F150 year to date""]","{""function_name"": ""get_average_sale_price"", ""arguments"": {""product_model"": ""Mustang"", ""date_range"": ""this_month""}}"
get_dealership_sales_summary,"Returns units sold, revenue, and average sale price grouped by dealership for a date range.","{""date_range"": {""type"": ""string"", ""required"": true, ""allowed_values"": [""this_month"", ""last_month"", ""year_to_date""]}}","[""Show dealership sales this month"", ""Dame ventas por dealership este mes"", ""Which dealership generated the most revenue year to date?""]","{""function_name"": ""get_dealership_sales_summary"", ""arguments"": {""date_range"": ""this_month""}}"


Function Registry created.
Available functions: ['get_product_sales_summary', 'get_top_selling_models', 'get_sales_by_rep', 'get_revenue_by_state', 'get_average_sale_price', 'get_dealership_sales_summary']
Available models: ['Accord', 'Camry', 'Civic', 'Explorer', 'F150', 'Mustang', 'RAV4', 'Silverado']


In [0]:
from datetime import datetime
import json

SOURCE_TABLE = "auto_sales_transactions"
FUNCTION_CATALOG_TABLE = "assistant_function_catalog"
AUDIT_TABLE = "assistant_query_audit"

print(f"Source table: {SOURCE_TABLE}")
print(f"Function catalog table: {FUNCTION_CATALOG_TABLE}")
print(f"Audit table: {AUDIT_TABLE}")

Source table: auto_sales_transactions
Function catalog table: assistant_function_catalog
Audit table: assistant_query_audit


Function name:
get_product_sales_summary

Description:
Returns units sold, total revenue, and average sale price for a specific vehicle model in a date range.

Arguments schema:
{
  "product_model": {
    "type": "string",
    "required": true,
    "allowed_values": [
      "Accord",
      "Camry",
      "Civic",
      "Explorer",
      "F150",
      "Mustang",
      "RAV4",
      "Silverado"
    ]
  },
  "date_range": {
    "type": "string",
    "required": true,
    "allowed_values": [
      "this_month",
      "last_month",
      "year_to_date"
    ]
  }
}

Example questions:
[
  "How many F150 trucks did I sell this month?",
  "Cuántas F150 he vendido este mes?",
  "What revenue did Silverado generate last month?",
  "Dame el resumen de ventas de Mustang este mes"
]

Expected JSON output:
{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

---

Function name:
get_top_selling_models

Description:
Retur

{
  "status": "success",
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  },
  "records": [
    {
      "product_model": "F150",
      "units_sold": 5,
      "total_revenue": 266500.0,
      "average_sale_price": 53300.0
    }
  ],
  "answer": "For this_month, F150 sold 5 units with $266,500.00 total revenue and an average sale price of $53,300.00.",
  "source_table": "auto_sales_transactions"
}


average_sale_price,product_model,total_revenue,units_sold
53300.0,F150,266500.0,5


Top selling models for this_month:
1. F150 (Ford): 5 units, $266,500.00
2. Silverado (Chevrolet): 2 units, $99,500.00
3. Explorer (Ford): 1 units, $47,000.00
4. Mustang (Ford): 1 units, $44,000.00
5. RAV4 (Toyota): 1 units, $39,000.00


...

{
  "status": "rejected",
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "next_century"
  },
  "error": "Invalid date_range 'next_century'. Allowed values: ['this_month', 'last_month', 'year_to_date']",
  "source_table": "auto_sales_transactions"
}



# Notebook result

We created a Python FunctionRegistry that executes safe business functions over Delta Tables.

## What the registry does

- loads allowed functions from assistant_function_catalog
- loads valid product models from auto_sales_transactions
- validates function names
- validates arguments
- resolves business date ranges into real dates
- executes controlled Spark queries
- returns structured responses

## Flow now

User question  
↓  
Foundation model will later return function JSON  
↓  
FunctionRegistry validates JSON  
↓  
FunctionRegistry executes safe method  
↓  
Spark queries auto_sales_transactions  
↓  
Answer is generated from Delta Table results  

## Why this reduces hallucinations

The model does not calculate sales numbers.  
The model does not write SQL.  
The model does not access arbitrary tables.  

Python controls execution.  
Delta Table is the source of truth.

## Next notebook

Notebook 05 will create the Foundation Model Router.

That router will convert natural language questions like:

¿Cuántas F150 vendí este mes?

into JSON like:

{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

Question: Cuántas F150 he vendido este mes?
Router output:
{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}
Question: Cuáles son mis modelos más vendidos este mes?
Router output:
{
  "function_name": "get_top_selling_models",
  "arguments": {
    "date_range": "this_month",
    "limit": 5
  }
}
Question: Qué vendedor vendió más este mes?
Router output:
{
  "function_name": "get_sales_by_rep",
  "arguments": {
    "date_range": "this_month"
  }
}
Question: Cuánto vendimos por estado este mes?
Router output:
{
  "function_name": "get_revenue_by_state",
  "arguments": {
    "date_range": "this_month"
  }
}
Question: Cuál fue el ticket promedio de Mustang este mes?
Router output:
{
  "function_name": "get_average_sale_price",
  "arguments": {
    "product_model": "Mustang",
    "date_range": "this_month"
  }
}
Question: Dame ventas por dealership este mes
Router output:
{
  "function_name": "get_dealership_

Router output:
{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

Validation:
{
  "is_valid": true,
  "error": null
}



# Notebook result

In this notebook we created a Foundation Model Router.

## What the router does

Input:

Natural language question

Output:

Structured function call JSON

Example:

{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

## Important

The router does not answer the business question.

The router does not calculate sales numbers.

The router does not generate SQL.

It only chooses a safe function and arguments.

## Flow now

User question  
↓  
Prompt Builder  
↓  
Databricks Foundation Model / ai_query  
↓  
JSON router output  
↓  
Validation  

## Next notebook

Notebook 06 will connect:

Router output  
↓  
FunctionRegistry  
↓  
Safe Delta query  
↓  
Final answer  
↓  
Audit log

In [0]:
# Instanciate Function Registry

registry = FunctionRegistry(
    spark=spark,
    source_table=SOURCE_TABLE,
    function_catalog_table=FUNCTION_CATALOG_TABLE,
    today=TODAY
)

print("FunctionRegistry ready.")
print("Available functions:")
print(registry.available_functions)

FunctionRegistry ready.
Available functions:
['get_product_sales_summary', 'get_top_selling_models', 'get_sales_by_rep', 'get_revenue_by_state', 'get_average_sale_price', 'get_dealership_sales_summary']


In [0]:
# Save audit log
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType
import json

AUDIT_SCHEMA = StructType([
    StructField("question", StringType(), True),
    StructField("router_output_json", StringType(), True),
    StructField("function_name", StringType(), True),
    StructField("arguments_json", StringType(), True),
    StructField("execution_status", StringType(), True),
    StructField("answer", StringType(), True),
    StructField("error", StringType(), True),
    StructField("records_json", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("created_at_utc", StringType(), True),
])

def save_audit_record(
    question: str,
    router_output: dict,
    execution_result: dict
) -> None:
    audit_record = [{
        "question": str(question or ""),
        "router_output_json": json.dumps(router_output or {}, ensure_ascii=False, default=str),
        "function_name": str((router_output or {}).get("function_name") or ""),
        "arguments_json": json.dumps((router_output or {}).get("arguments", {}), ensure_ascii=False, default=str),
        "execution_status": str((execution_result or {}).get("status") or ""),
        "answer": str((execution_result or {}).get("answer") or ""),
        "error": str((execution_result or {}).get("error") or ""),
        "records_json": json.dumps((execution_result or {}).get("records", []), ensure_ascii=False, default=str),
        "source_table": str((execution_result or {}).get("source_table") or SOURCE_TABLE),
        "created_at_utc": datetime.now(timezone.utc).isoformat()
    }]

    audit_df = spark.createDataFrame(audit_record, schema=AUDIT_SCHEMA)

    audit_df.write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

    print(f"Audit record saved to {AUDIT_TABLE}")


In [0]:
# Assistant main function

def answer_business_question(question: str, prefer_ai_query: bool = True) -> dict:
    router_output = route_question(
        user_question=question,
        prefer_ai_query=prefer_ai_query
    )

    validation_result = validate_router_output(router_output)

    if not validation_result["is_valid"]:
        execution_result = {
            "status": "rejected",
            "function_name": router_output.get("function_name"),
            "arguments": router_output.get("arguments", {}),
            "answer": "The router output was invalid.",
            "error": validation_result["error"],
            "records": [],
            "source_table": SOURCE_TABLE
        }

        save_audit_record(question, router_output, execution_result)
        return {
            "question": question,
            "router_output": router_output,
            "execution_result": execution_result
        }

    function_name = router_output["function_name"]
    arguments = router_output["arguments"]

    if function_name == "unsupported":
        execution_result = {
            "status": "rejected",
            "function_name": function_name,
            "arguments": arguments,
            "answer": "This question is outside the supported Acme Motors function catalog.",
            "error": arguments.get("reason"),
            "records": [],
            "source_table": SOURCE_TABLE
        }

        save_audit_record(question, router_output, execution_result)
        return {
            "question": question,
            "router_output": router_output,
            "execution_result": execution_result
        }

    execution_result = registry.execute(
        function_name=function_name,
        arguments=arguments
    )

    save_audit_record(question, router_output, execution_result)

    return {
        "question": question,
        "router_output": router_output,
        "execution_result": execution_result
    }

In [0]:
# Test F150 question
result = answer_business_question(
    "Cuántas F150 he vendido este mes?",
    prefer_ai_query=True
)

print("Question:")
print(result["question"])

print("\nRouter output:")
print(json.dumps(result["router_output"], indent=2, ensure_ascii=False))

print("\nFinal answer:")
print(result["execution_result"]["answer"])

print("\nStatus:")
print(result["execution_result"]["status"])

Audit record saved to assistant_query_audit
Question:
Cuántas F150 he vendido este mes?

Router output:
{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

Final answer:
For this_month, F150 sold 5 units with $266,500.00 total revenue and an average sale price of $53,300.00.

Status:
success


In [0]:
records = result["execution_result"].get("records", [])

if records:
    display(spark.createDataFrame(records))
else:
    print("No records returned.")

average_sale_price,product_model,total_revenue,units_sold
53300.0,F150,266500.0,5


In [0]:
# Test multiple questions
questions = [
    "Cuántas F150 he vendido este mes?",
    "Cuáles son mis modelos más vendidos este mes?",
    "Qué vendedor vendió más este mes?",
    "Cuánto vendimos por estado este mes?",
    "Cuál fue el ticket promedio de Mustang este mes?",
    "Dame ventas por dealership este mes",
    "Recomiéndame una laptop gamer"
]

for question in questions:
    print("=" * 100)
    output = answer_business_question(question, prefer_ai_query=True)

    print("Question:")
    print(output["question"])

    print("\nRouter output:")
    print(json.dumps(output["router_output"], indent=2, ensure_ascii=False))

    print("\nAnswer:")
    print(output["execution_result"]["answer"])

    print("\nStatus:")
    print(output["execution_result"]["status"])

Audit record saved to assistant_query_audit
Question:
Cuántas F150 he vendido este mes?

Router output:
{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

Answer:
For this_month, F150 sold 5 units with $266,500.00 total revenue and an average sale price of $53,300.00.

Status:
success
Audit record saved to assistant_query_audit
Question:
Cuáles son mis modelos más vendidos este mes?

Router output:
{
  "function_name": "get_top_selling_models",
  "arguments": {
    "date_range": "this_month",
    "limit": 5
  }
}

Answer:
Top selling models for this_month:
1. F150 (Ford): 5 units, $266,500.00
2. Silverado (Chevrolet): 2 units, $99,500.00
3. Explorer (Ford): 1 units, $47,000.00
4. Mustang (Ford): 1 units, $44,000.00
5. RAV4 (Toyota): 1 units, $39,000.00

Status:
success
Audit record saved to assistant_query_audit
Question:
Qué vendedor vendió más este mes?

Router output:
{
  "function_name": "get_sales_b

In [0]:
# Read audit table

audit_df = spark.table(AUDIT_TABLE)

display(
    audit_df.select(
        "question",
        "function_name",
        "arguments_json",
        "execution_status",
        "answer",
        "source_table",
        "created_at_utc"
    ).orderBy("created_at_utc", ascending=False)
)

question,function_name,arguments_json,execution_status,answer,source_table,created_at_utc
Recomiéndame una laptop gamer,unsupported,"{""reason"": ""Question is outside the supported function catalog.""}",rejected,This question is outside the supported Acme Motors function catalog.,auto_sales_transactions,2026-05-24T08:33:34.104997+00:00
Dame ventas por dealership este mes,get_dealership_sales_summary,"{""date_range"": ""this_month""}",success,"Dealership sales summary for this_month: Acme Motors North: 5 units, $266,500.00, avg sale price $53,300.00 Acme Motors South: 3 units, $130,500.00, avg sale price $43,500.00 Acme Motors East: 3 units, $119,000.00, avg sale price $39,666.67 Acme Motors West: 1 units, $39,000.00, avg sale price $39,000.00",auto_sales_transactions,2026-05-24T08:33:31.144529+00:00
Cuál fue el ticket promedio de Mustang este mes?,get_average_sale_price,"{""product_model"": ""Mustang"", ""date_range"": ""this_month""}",success,"For this_month, Mustang had 1 sales with an average sale price of $44,000.00. Min: $44,000.00, Max: $44,000.00.",auto_sales_transactions,2026-05-24T08:33:27.759228+00:00
Cuánto vendimos por estado este mes?,get_revenue_by_state,"{""date_range"": ""this_month""}",success,"Revenue by state for this_month: TX: 5 units, $266,500.00 CA: 3 units, $130,500.00 AZ: 2 units, $91,000.00 WA: 1 units, $39,000.00 NV: 1 units, $28,000.00",auto_sales_transactions,2026-05-24T08:33:23.931493+00:00
Qué vendedor vendió más este mes?,get_sales_by_rep,"{""date_range"": ""this_month""}",success,"Sales by representative for this_month: Ana Lopez: 5 units, $266,500.00 Luis Garcia: 3 units, $130,500.00 Mike Johnson: 2 units, $91,000.00 Carlos Rivera: 1 units, $39,000.00 Sarah Kim: 1 units, $28,000.00",auto_sales_transactions,2026-05-24T08:33:20.547238+00:00
Cuáles son mis modelos más vendidos este mes?,get_top_selling_models,"{""date_range"": ""this_month"", ""limit"": 5}",success,"Top selling models for this_month: 1. F150 (Ford): 5 units, $266,500.00 2. Silverado (Chevrolet): 2 units, $99,500.00 3. Explorer (Ford): 1 units, $47,000.00 4. Mustang (Ford): 1 units, $44,000.00 5. RAV4 (Toyota): 1 units, $39,000.00",auto_sales_transactions,2026-05-24T08:33:17.106324+00:00
Cuántas F150 he vendido este mes?,get_product_sales_summary,"{""product_model"": ""F150"", ""date_range"": ""this_month""}",success,"For this_month, F150 sold 5 units with $266,500.00 total revenue and an average sale price of $53,300.00.",auto_sales_transactions,2026-05-24T08:33:13.548684+00:00
Cuántas F150 he vendido este mes?,get_product_sales_summary,"{""product_model"": ""F150"", ""date_range"": ""this_month""}",success,"For this_month, F150 sold 5 units with $266,500.00 total revenue and an average sale price of $53,300.00.",auto_sales_transactions,2026-05-24T08:29:10.212812+00:00
Cuántas F150 he vendido este mes?,get_product_sales_summary,"{""product_model"": ""F150"", ""date_range"": ""this_month""}",success,"For this_month, F150 sold 5 units with $266,500.00 total revenue and an average sale price of $53,300.00.",auto_sales_transactions,2026-05-24T08:27:39.785859+00:00


In [0]:
%sql
-- Get total audit by status
SELECT
  execution_status,
  COUNT(*) AS total_requests
FROM assistant_query_audit
GROUP BY execution_status;

execution_status,total_requests
success,8
rejected,1


In [0]:
%sql
-- Most used functions
SELECT
  function_name,
  COUNT(*) AS total_calls
FROM assistant_query_audit
GROUP BY function_name
ORDER BY total_calls DESC;

function_name,total_calls
get_product_sales_summary,3
get_dealership_sales_summary,1
get_top_selling_models,1
get_sales_by_rep,1
get_revenue_by_state,1
get_average_sale_price,1
unsupported,1


In [0]:
%sql
-- Rejected questions
SELECT
  question,
  function_name,
  error,
  created_at_utc
FROM assistant_query_audit
WHERE execution_status = 'rejected'
ORDER BY created_at_utc DESC;

question,function_name,error,created_at_utc
Recomiéndame una laptop gamer,unsupported,Question is outside the supported function catalog.,2026-05-24T08:33:34.104997+00:00


In [0]:
%sql

SELECT
  question,
  router_output_json,
  execution_status
FROM assistant_query_audit
ORDER BY created_at_utc DESC;

question,router_output_json,execution_status
Recomiéndame una laptop gamer,"{""function_name"": ""unsupported"", ""arguments"": {""reason"": ""Question is outside the supported function catalog.""}}",rejected
Dame ventas por dealership este mes,"{""function_name"": ""get_dealership_sales_summary"", ""arguments"": {""date_range"": ""this_month""}}",success
Cuál fue el ticket promedio de Mustang este mes?,"{""function_name"": ""get_average_sale_price"", ""arguments"": {""product_model"": ""Mustang"", ""date_range"": ""this_month""}}",success
Cuánto vendimos por estado este mes?,"{""function_name"": ""get_revenue_by_state"", ""arguments"": {""date_range"": ""this_month""}}",success
Qué vendedor vendió más este mes?,"{""function_name"": ""get_sales_by_rep"", ""arguments"": {""date_range"": ""this_month""}}",success
Cuáles son mis modelos más vendidos este mes?,"{""function_name"": ""get_top_selling_models"", ""arguments"": {""date_range"": ""this_month"", ""limit"": 5}}",success
Cuántas F150 he vendido este mes?,"{""function_name"": ""get_product_sales_summary"", ""arguments"": {""product_model"": ""F150"", ""date_range"": ""this_month""}}",success
Cuántas F150 he vendido este mes?,"{""function_name"": ""get_product_sales_summary"", ""arguments"": {""product_model"": ""F150"", ""date_range"": ""this_month""}}",success
Cuántas F150 he vendido este mes?,"{""function_name"": ""get_product_sales_summary"", ""arguments"": {""product_model"": ""F150"", ""date_range"": ""this_month""}}",success



# Notebook result

We completed the end-to-end Acme Motors Assistant.

## Final architecture

Natural language question  
↓  
Databricks Foundation Model through ai_query  
↓  
Function call JSON  
↓  
Python validation  
↓  
FunctionRegistry  
↓  
Safe Spark query  
↓  
auto_sales_transactions Delta Table  
↓  
Final answer  
↓  
assistant_query_audit Delta Table  

## What this demonstrates

- Databricks notebooks
- Delta Tables as source of truth
- SQL over business data
- Foundation Model routing
- Prompt-as-function design
- Python OOP with FunctionRegistry
- Safe query execution
- Argument validation
- Audit logging
- Hallucination control

## Key design decision

The LLM routes.  
Python executes.  
Delta is the source of truth.

## Next notebook

Notebook 07 will focus on quality checks, debugging, and reviewing the audit log.